# Class 3 — Building a Simple RAG Chatbot

**Week 6: Foundations of RAG and Chatbots**

### Learning objectives
By the end of this notebook you will be able to:
- Chunk a small set of documents into retrieval-sized pieces
- Embed chunks and store them in a simple in-memory structure
- Retrieve the top-k most relevant chunks for a question using cosine similarity
- Construct a grounded prompt from retrieved chunks and generate a final answer
- Name the main limitations of this simple pipeline and what production systems add on top

> This is the capstone notebook for Week 6: everything from Class 1 (grounding) and Class 2 (embeddings and
> similarity) comes together into one working, minimal RAG chatbot.

## Setup

In [ ]:
import os
import numpy as np

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not OPENAI_API_KEY:
    print(
        "No OPENAI_API_KEY found in your environment.\n"
        "This notebook uses OpenAI's embeddings endpoint for retrieval. Set it with:\n"
        "  export OPENAI_API_KEY=sk-...\n"
        "The generation step below can use either OPENAI_API_KEY or ANTHROPIC_API_KEY."
    )
else:
    print("Found OPENAI_API_KEY -- retrieval demo is ready to run.")

if not OPENAI_API_KEY and not ANTHROPIC_API_KEY:
    print("No key at all found for generation either -- set one of the two before running the full pipeline.")
elif ANTHROPIC_API_KEY:
    print("Found ANTHROPIC_API_KEY -- it will be used for the generation step.")
else:
    print("Will use OPENAI_API_KEY for the generation step too.")

In [ ]:
def call_llm(prompt, system=None, model_openai="gpt-4o-mini", model_anthropic="claude-3-5-haiku-20241022"):
    """Send a single prompt to whichever provider has a key configured. Returns a plain string reply."""
    if ANTHROPIC_API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        response = client.messages.create(
            model=model_anthropic,
            max_tokens=400,
            system=system or "You are a helpful, concise assistant.",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text
    elif OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        response = client.chat.completions.create(model=model_openai, messages=messages)
        return response.choices[0].message.content
    else:
        raise RuntimeError("No API key configured. Set OPENAI_API_KEY or ANTHROPIC_API_KEY and re-run Setup.")


def get_embedding(text, model="text-embedding-3-small"):
    """Return the embedding vector (a list of floats) for a piece of text. Requires OPENAI_API_KEY."""
    from openai import OpenAI
    if not OPENAI_API_KEY:
        raise RuntimeError("get_embedding requires OPENAI_API_KEY. Set it and re-run Setup.")
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding


def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## Concept 1 — Our Tiny Document Set

A real RAG system starts with a pile of documents (PDFs, wiki pages, tickets). Ours is three short plain-text
"policy" documents, kept intentionally small so the whole pipeline is easy to trace end to end.

In [ ]:
raw_documents = [
    """Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting
    from their first day. Unused days roll over up to a maximum of 5 days into the next year.""",

    """Expense Policy: Employees must submit expense reports within 30 days of purchase using the
    finance portal. Reports missing a receipt over $25 will be returned for correction.""",

    """Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and 3pm
    in their local time zone, and are expected to attend the weekly all-hands meeting on video.""",
]

for i, doc in enumerate(raw_documents, start=1):
    print(f"--- Document {i} ---")
    print(doc.strip())
    print()

## Concept 2 — Chunking

Our documents are already short, so each one becomes a single chunk. For a longer document you would split it
into paragraph-sized pieces first (see Class 3's slide deck for chunking strategies), then treat every chunk the
same way we're about to treat these three documents. We'll still write `chunk_documents` as a real, general
function so this notebook works if you paste in longer documents later.

In [ ]:
def chunk_text(text, max_chars=280, overlap=40):
    """Split text into chunks of roughly max_chars characters, with a small overlap between chunks
    so an idea isn't cut in half at a boundary.
    """
    text = " ".join(text.split())  # normalize whitespace
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start = end - overlap  # step back a little so chunks overlap
    return chunks


def chunk_documents(documents):
    """Chunk every document and return a flat list of {"text": ..., "source": ...} dicts."""
    chunks = []
    for doc_id, doc in enumerate(documents, start=1):
        for chunk in chunk_text(doc):
            chunks.append({"text": chunk, "source": f"Document {doc_id}"})
    return chunks

chunks = chunk_documents(raw_documents)
print(f"Produced {len(chunks)} chunk(s):\n")
for c in chunks:
    print(f"[{c['source']}] {c['text']}")

## Concept 3 — Embed and Store

We embed every chunk once and keep the vectors in a simple in-memory list of dictionaries. This *is* a vector
store — just a tiny, unindexed one. Chroma, FAISS, and Pinecone (Class 2) do this same job at a much larger
scale, with real indexing for speed.

In [ ]:
def build_index(chunks):
    """Embed every chunk and return a list of {"text", "source", "embedding"} dicts -- our mini vector store."""
    index = []
    for c in chunks:
        index.append({
            "text": c["text"],
            "source": c["source"],
            "embedding": get_embedding(c["text"]),
        })
    return index

vector_store = build_index(chunks)
print(f"Indexed {len(vector_store)} chunk(s) into the in-memory vector store.")

## Concept 4 — Retrieve Top-k

Given a question, embed it and rank every stored chunk by cosine similarity. Keep the top `k` — usually 2 to 5
in a real system — as the context we'll hand to the LLM.

In [ ]:
def retrieve(question, vector_store, k=2):
    """Return the top-k chunks (as dicts) from vector_store, ranked by cosine similarity to the question."""
    query_vec = get_embedding(question)
    scored = [
        (cosine_similarity(query_vec, item["embedding"]), item)
        for item in vector_store
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [item for _, item in scored[:k]]

question = "How many vacation days do new hires get?"
top_chunks = retrieve(question, vector_store, k=2)

for rank, chunk in enumerate(top_chunks, start=1):
    print(f"{rank}. [{chunk['source']}] {chunk['text']}")

## Concept 5 — Construct a Grounded Prompt and Generate

Now we build the prompt exactly the way we did by hand in Class 1 — except the context is retrieved automatically
instead of pasted by us.

In [ ]:
def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(f"[{c['source']}] {c['text']}" for c in retrieved_chunks)
    return f"""Answer using ONLY the context below. If the answer isn't in the context, say you don't know.

CONTEXT:
{context}

QUESTION: {question}
"""

prompt = build_prompt(question, top_chunks)
print(prompt)

In [ ]:
answer = call_llm(prompt)
print(answer)

## Concept 6 — The Whole Pipeline, One Function

Let's wrap everything above into a single `rag_answer` function, so you can ask any question against this tiny
knowledge base in one call.

In [ ]:
def rag_answer(question, vector_store, k=2):
    """Run the full retrieve -> construct prompt -> generate pipeline for a single question."""
    retrieved = retrieve(question, vector_store, k=k)
    prompt = build_prompt(question, retrieved)
    return call_llm(prompt), retrieved

answer, sources = rag_answer("Do remote employees need to be on camera for meetings?", vector_store)
print("ANSWER:", answer)
print("\nSOURCES USED:")
for s in sources:
    print(f"  - [{s['source']}] {s['text'][:80]}...")

## Limitations of This Simple Pipeline

- **Retrieval quality ceiling** — if the wrong chunk is retrieved, the answer will be wrong no matter how good the
  LLM is.
- **No built-in citations** — we printed sources manually above; a real product needs to surface this to users.
- **Stale index** — if `raw_documents` changes, you must re-run `build_index` or the answers go out of date.
- **Cost and latency stack up** — every question triggers one embedding call and one generation call.
- **Doesn't replace fine-tuning** — this pipeline is about facts, not tone or behavior.

Next steps beyond this notebook: re-ranking, hybrid (keyword + semantic) search, metadata filtering, real
citations, and systematic evaluation of retrieval and answer quality.

## Challenges

Each of these builds directly on `chunk_documents`, `build_index`, `retrieve`, and `rag_answer` above.

### Challenge 1 — Swap In Your Own Documents

Replace `raw_documents` with three short documents about a topic you know well (a hobby, a class, a project). Ask
a new question and print the answer along with which source(s) were used.

**Acceptance criteria:** your printed answer is correct according to your own documents, and cites the right
source.

In [ ]:
# TODO: write 3 new documents, rebuild the index, and ask a question about them


### Challenge 2 — Change the Chunk Size

Re-run `chunk_text` with a much smaller `max_chars` (e.g. 60) on one of the original documents and print the
resulting chunks. Then retrieve for the vacation-days question again using this smaller chunking and observe
whether the top retrieved chunk changes.

**Acceptance criteria:** you print the chunks produced by both chunk sizes and state in a comment whether
retrieval quality got better, worse, or stayed the same.

In [ ]:
# TODO: re-chunk with a smaller max_chars, rebuild the index, and compare retrieval results


### Challenge 3 — Add a Real Citation

Modify `rag_answer` (or write a new version) so the returned answer string includes an inline citation like
`[Document 1]` after any sentence that uses that source's information.

**Acceptance criteria:** your function's output visibly includes at least one `[Document N]`-style citation.

In [ ]:
# TODO: modify or extend rag_answer to include inline citations in the returned text


### Challenge 4 — Force a "Don't Know"

Ask `rag_answer` a question that your three documents genuinely do not cover (e.g. something about a topic never
mentioned). Confirm the model says it doesn't know rather than guessing.

**Acceptance criteria:** you print the question and answer, and the answer explicitly states the information
isn't available, rather than fabricating a plausible-sounding response.

In [ ]:
# TODO: ask a question your documents don't cover and confirm the model admits it doesn't know


### Challenge 5 — Bonus: Add a Conflicting Document

Add a fourth document that directly contradicts one of the first three (e.g. a different, later "policy update"
stating a different number of vacation days). Ask the original question again and observe what the pipeline
retrieves and answers.

**Acceptance criteria:** you print what got retrieved and the final answer, and add a comment explaining, in your
own words, why this is a real limitation of simple RAG systems.

In [ ]:
# TODO: add a conflicting fourth document, re-run the pipeline, and comment on what happens
